In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import json
os.chdir("..")

In [4]:
from sklearn.metrics import f1_score
from torch.nn import CrossEntropyLoss
from torch.utils.data import DataLoader
from torch.optim import Adam
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import Dataset
from pathlib import Path
from argparse import ArgumentParser

import os
import ray
import time
import torch
import json
import torchmetrics
import numpy as np
import datasets

import torchmetrics.aggregation
from transformers import AutoTokenizer
from tqdm.auto import tqdm
from datasets import load_dataset


from cluster_intrep_repo.utils import initialize_tokenizer, tokenize_blocksworld_generation, THINK_TOKEN
from cluster_intrep_repo.stacks_utils import *
from tqdm.auto import tqdm, trange
from more_itertools import chunked

In [5]:
ray.init(address="auto", namespace="blocksworld")


2025-03-27 01:26:31,629	INFO worker.py:1654 -- Connecting to existing Ray cluster at address: 10.61.4.10:6379...
2025-03-27 01:26:31,640	INFO worker.py:1832 -- Connected to Ray cluster. View the dashboard at http://127.0.0.1:8265 


Python version:,3.11.11
Ray version:,2.42.1
Dashboard:,http://127.0.0.1:8265


(collect_section_tokens_remote pid=634944) Collecting section tokens
(raylet) The autoscaler failed with the following error:
Terminated with signal 15
  File "/home/nebius/openr1/lib/python3.11/site-packages/ray/autoscaler/_private/monitor.py", line 719, in <module>
    monitor.run()
  File "/home/nebius/openr1/lib/python3.11/site-packages/ray/autoscaler/_private/monitor.py", line 604, in run
    self._run()
  File "/home/nebius/openr1/lib/python3.11/site-packages/ray/autoscaler/_private/monitor.py", line 458, in _run
    time.sleep(AUTOSCALER_UPDATE_INTERVAL_S)



In [6]:
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

compute_dtype = torch.bfloat16
device = 'cuda'
model_id = "Qwen/QwQ-32B"

tokenizer = initialize_tokenizer(model_id)


In [7]:
n_blocks = 6


cur_dir = Path(".").absolute()


def load_dataset_from_file(domain_name, task_name):
    prompt_dir = cur_dir / Path(f"./cot-planning/results/{domain_name}/deepseek-32b/")
    with open(prompt_dir / f"{task_name}.json", 'r') as file:
        return json.load(file)

def map_sections_into_tokens(sections: list[tuple[str, str]], row: dict) -> list[dict]:
    # tokens = tokenize_blocksworld_generation(tokenizer, row)
    generation = row["generation"]
    section_tokens = []
    for label, content in sections[1:]:
        text_pos = generation.find(content[:300])
        if text_pos == -1:
            continue
        text_before = generation[:text_pos]
        tokens_before = tokenize_blocksworld_generation(tokenizer, row, text_before)[0, :-2]
        
        if len(tokens_before) > 6000:
            continue
        
        content_tokens = tokenizer.encode(" " + content)[:-2]
        
        section_tokens.append({
            "label": label,
            "pos_before": len(tokens_before),
            "pos_after": len(tokens_before) + len(content_tokens),
            "text_pos": text_pos,
            "content": content
        })

    return section_tokens

import re

labels_low = [
    "initial-state-understanding",
    "goal-state-understanding",
    "state-tracking",
    "action-exploration",
    "state-tracking"
]

def parse_sections(text:str, labels: list[str]) -> list[tuple[str, str]]:
    pattern = r'\["([^"]+)"\](.*?)\["end-section"\]'
    matches = re.findall(pattern, text, re.DOTALL)
    
    sections = []
    for label, content in matches:
        if label in labels:
            sections.append((label, content.strip()))
    
    return sections

@ray.remote
def collect_section_tokens_remote(labels_dataset: dict, dataset: dict, sections):
    section_tokens = {}
    print("Collecting section tokens")
    for idx in dataset:
        label = labels_dataset[idx]["label"]
        if label is None:
            continue
        sections = parse_sections(labels_dataset[idx]["label"], labels_low)
        section_tokens[idx] = map_sections_into_tokens(sections, dataset[idx])
        
    return section_tokens

def collect_section_tokens(labels_dataset: datasets.Dataset, dataset: datasets.Dataset, labels_low: list[str], n_rows: int, n_workers: int):
    section_tokens = {}
    futures = []
    rows_per_worker = n_rows // n_workers
    
    idx_to_row = {i: x for i, x in enumerate(dataset)}
    idx_to_label = {i: x for i, x in enumerate(labels_dataset)}
    
    for i in range(n_workers):
        start_idx = i * rows_per_worker
        end_idx = (i + 1) * rows_per_worker
        futures.append(collect_section_tokens_remote.remote(
            {
                k: idx_to_label[k] for k in range(start_idx, end_idx)    
            }, 
            {
                k: idx_to_row[k] for k in range(start_idx, end_idx)  
            },
            labels_low,
        ))

    for future in futures:
        section_tokens.update(ray.get(future))

    return section_tokens
    

def load_datasets():
    dataset = load_dataset(
    f"dmitriihook/qwq-32b-planning-6-blocks")["train"]
    
    task_name = "plan_generation_po"
    domain_name = f"blocksworld_{n_blocks}_blocks"
    eval_results = load_dataset_from_file(domain_name, task_name)["instances"]
    eval_results = {x["dataset_idx"]: x for x in eval_results}

    labels_dataset = load_dataset("dmitriihook/blocksworld-6-blocks-qwq-reasoning-parts-low-v4")["train"]
    
    section_tokens = collect_section_tokens(labels_dataset, dataset, labels_low, len(labels_dataset), 32)

    return dataset, eval_results, section_tokens


# total_layers = model.config.num_hidden_layers
total_layers = 64


def make_data_to_process(dataset, section_tokens, n_rows, eval_results, answer_type, tokenizer):
    data_to_process = []
    for idx, row in enumerate(dataset.select(range(n_rows))):
        if eval_results[idx]["llm_correct"] and answer_type == "incorrect":
            continue
        if not eval_results[idx]["llm_correct"] and answer_type == "correct":
            continue
        generation = row["generation"]

        if "[PLAN]\n" not in generation:
            continue

        if idx not in section_tokens:
            continue
        
        actions = extract_actions(row)
        if actions is None:
            continue

        parsed_actions = parse_block_actions(actions)
        plan_start = generation.index("[PLAN]\n") + len("[PLAN]\n")
        counts = defaultdict(int)

        for section in section_tokens[idx]:
            if section["pos_after"] < section["pos_before"] + 2:
                continue
            
            counts[section["label"]] += 1
                
            data_to_process.append({
                "idx": idx,
                "pos_before": section["pos_before"],
                "pos_after": section["pos_after"],
                "actions": parsed_actions,
                "label": section["label"],
                "label_n": counts[section["label"]],
            })


    return data_to_process


def process_data(items, n_blocks: int, tgt_action: int, part_type: str, min_n: int, max_n: int, max_tokens = 3200):
    new_items = []

    for item in items:
        actions = item["actions"]
        if item["label"] != part_type:
            continue
        if item["label_n"] < min_n or item["label_n"] > max_n:
            continue
        try:
            blocks = actions[tgt_action][1]
            block = blocks[0]
            label = block2int(block, n_blocks)
        except Exception as e:
            continue

        new_items.append({
            "pos_before": item["pos_before"],
            "label": label,
            "idx":  item["idx"],
        })

    return new_items
    

def collate_fn(batch):
    inputs = [torch.tensor(x, dtype=compute_dtype) for x in batch["input"]]    
    masks = [torch.ones(x.shape[0], dtype=torch.bool) for x in inputs]
    inputs = pad_sequence(inputs, batch_first=True,
                          padding_value=0, padding_side="left")
    masks = pad_sequence(masks, batch_first=True,
                         padding_value=True, padding_side="left")

    # inputs = torch.stack(inputs)
    # masks = torch.stack(masks)

    labels = np.stack([x for x in batch["labels"]])
    labels = torch.tensor(labels, dtype=torch.int64)

    return {
        "input": inputs,
        "labels": labels,
        "mask": masks
    }

@ray.remote
class CollateActor:
    def __init__(self):
        pass

    def collate_fn(self, batch):
        return collate_fn(batch)


In [9]:
class StepProbeDataset(Dataset):
    def __init__(self, items, n_layer, n_prev_tokens, shift_tokens, hidden_states, n_blocks, batch_size, tgt_action, part_type, min_n, max_n):
        self.items = process_data(items, n_blocks, tgt_action, part_type, min_n, max_n)
        self.n_layer = n_layer
        self.n_blocks = n_blocks
        self.n_prev_tokens = n_prev_tokens
        self.shift_tokens = shift_tokens
        self.hidden_states = hidden_states
        self.batch_size = batch_size

    def get_batch(self, idxs):
        items = [self.items[idx] for idx in idxs]
        refs = [self.hidden_states[item["idx"]][self.n_layer] for item in items]
        _hidden_states = ray.get(refs)

        inputs = []
        labels = []

        for i, item in enumerate(items):
            pos_start = item["pos_before"]

            pos = pos_start

            window_start = max(0, pos - self.n_prev_tokens - self.shift_tokens)
            window_end = pos - self.shift_tokens + 1


            label = item["label"]
            hidden_states = _hidden_states[i]

            inputs.append(hidden_states[window_start:window_end])
            labels.append(label)
        return {
            "input": inputs,
            "labels": labels,
        }
        

    def __len__(self):
        return len(self.items) // self.batch_size
    
    def __getitem__(self, idx):
        start_item_idx = idx * self.batch_size
        end_item_idx = min((idx + 1) * self.batch_size, len(self.items))

        batch = self.get_batch(range(start_item_idx, end_item_idx))

        batch = collate_fn(batch)

        return batch


In [10]:
dataset, eval_results, section_tokens = load_datasets()

In [12]:

training_data = make_data_to_process(dataset, section_tokens, 2000, eval_results, "all", tokenizer)

n_train = int(len(training_data) * 0.9)

train_items = training_data[:n_train]
test_items = training_data[n_train:]


In [13]:
actor_handle = ray.get_actor("dataset_actor")

In [79]:
hidden_states = ray.get(actor_handle.get_hidden_states.remote())
    
train_dataset = StepProbeDataset(
train_items, 39, 41, -40, hidden_states, n_blocks, 64, 0, "action-exploration", 1, 1)

In [114]:
item = train_dataset.items[5]

In [115]:
pos_start = item["pos_before"]

pos = pos_start

window_start = max(0, pos - train_dataset.n_prev_tokens - train_dataset.shift_tokens)
window_end = pos - train_dataset.shift_tokens + 1


In [116]:
window_start, window_end

(2351, 2393)

In [117]:
tokens = tokenize_blocksworld_generation(tokenizer, dataset[item["idx"]], dataset[item["idx"]]["generation"])   

In [118]:
print(tokenizer.decode(tokens[0, window_start:window_end]))

 

So first step: unstack D from A. 

After unstacking D, D is now in hand, and A is now clear. 

Then, we can put D down somewhere. But where


In [119]:
print(tokenizer.decode(tokens[0, window_start-200:window_end + 100]))

 but D is currently on top of A. To get D onto E, we need to move D down. But D is on top of A, so to move D, we need to unstack it from A. But to do that, A must be clear. However, currently, D is on A, so A is not clear. Wait, but the initial conditions say that D is clear. Wait, the initial conditions state:

"Block D is clear" — so D has no blocks on top of it. Wait, but in the initial stack, D is on top of A, so A has D on it, so A is not clear. But D is clear because nothing is on top of it. 

So to unstack D from A, we can do that because D is clear. 

Wait, the action "unstack a block from on top of another block" requires that the block being unstacked is clear. Since D is clear, that's okay. 

So first step: unstack D from A. 

After unstacking D, D is now in hand, and A is now clear. 

Then, we can put D down somewhere. But where? The goal is to have D on E. 

Wait, E is on the table. So to stack D on E, we need to stack D onto E. 

But first, we need to have D in hand. 

Wa